# 第八週作業：使用 RAG 技術打造個人化角色對話機器人

## 📌 作業目標

本週目標為實作 Retrieval-Augmented Generation（RAG）系統，並透過語意檢索與語言模型整合，讓對話機器人能夠回應與資料庫相關的自然語言提問。我以「哈利波特」世界觀延伸出來的虛構角色為基礎，打造一個具有風格與個性的人設問答系統。

---

## 👤 人設 / 背景設定

我創造了 4 位虛構角色並撰寫其個人設定，儲存為文字檔後餵入語意向量資料庫，作為機器人的知識基礎。角色如下：

- **葵崎 凜**（Ravenclaw 學生，溫柔靜謐，最愛奇獸與魔藥）
- **梅里・米勒**（Hufflepuff 學生，開朗樂觀，夢想成為魔法治癒師）
- **嶺夜 綾人**（Slytherin 學生，冷靜理性，擅長防禦與符文）
- **諾亞・席洛**（Gryffindor 學生，神秘安靜，研究魔法星象）

系統扮演的是「麥米奈娃·麥教授（Minerva McGonagall）」，她會以教授的身份回應問題，並且認知「葵崎凜是她最得意的學生」。

---

## 📚 使用的資料（RAG新增內容）

我將上述 4 位角色的 `.txt` 描述檔餵入向量資料庫中，並使用 `sentence-transformers/all-MiniLM-L6-v2` 建立語意嵌入（embeddings），透過 FAISS 建構向量資料庫。回答時，系統會根據問題使用 `.similarity_search()` 擷取相關段落並融合進 prompt，由 Groq 的 LLaMA3 模型輸出自然語言回應。

---

## ⚠️ 系統觀察：正確 vs 幻覺

### ✅ 成功案例：
- 問題：「葵崎凜的最好的朋友是誰？」  
  回答：「是露娜·羅古德（Luna Lovegood）！」

- 問題：「她最喜歡哪堂課？」  
  回答：「魔藥學與奇獸飼育學，尤其喜歡變形學」

### ❌ 幻覺案例：
- 問題：「葵崎凜的貓叫什麼？」  
  回答：「不知道名字，但可能是 Mocha？」或「沒有資料」  
  > 實際上資料庫明確記載：貓叫「Mocha」

- 問題：「她最喜歡的教授是誰？」  
  回答：出現一段混合中英文的說法：「which is none other than myself, Minerva McGonagall」

---

## 🤔 為什麼會產生幻覺？

1. **context 未命中**：問題未觸發正確段落（語意搜尋落空）
2. **模型自由發揮**：LLaMA3 訓練時會傾向補完空白，自動腦補資訊
3. **prompt 語意不夠限制**：語言模型無法判斷是否「只能回答 context 中有的資訊」

---

## 🔒 強化處理

我設計了一套「防幻覺 prompt」：

- 明確指示：「如果資料中沒有就說不知道」
- 禁止對提問者做出稱謂（避免誤叫「梅里・米勒」）
- context 直接塞進 user prompt，而非單純用作系統背景

---
## 📌 心得

這份作業不只是技術實作，更讓我體驗了 RAG 系統在人設模擬、世界觀延伸與語意推理方面的潛力。雖然模型仍可能產生幻覺，但透過 prompt 工程與向量設計，可以大幅提升準確率。未來我希望加入更多角色與互動維度，打造出真正有「靈魂」的角色型 AI 對話系統。


# 第一步：向量化角色資料
##🧩 說明：
- 目的：自動讀取多個角色人設檔案（.txt 格式），並透過向量化技術建立一個可以用於 RAG（Retrieval-Augmented Generation）系統的FAISS 向量資料庫。

- 這些角色資料來自原創設定，在《哈利波特》世界觀下的虛構角色，包含主角「葵崎凜」與其他三位配角。
- 資料將經過文字分割與語意嵌入處理，最終建立成一個可壓縮、可查詢的語意知識庫。

---
🎯 目標：
✅ 自動讀取資料夾中的所有 .txt 角色描述檔案

✅ 使用 LangChain 的 TextLoader 與 TextSplitter 將長文本切段處理

✅ 使用 sentence-transformers 模型將文本嵌入為向量

✅ 建立 FAISS 向量資料庫並儲存為 faiss_db.zip，供後續對話系統使用

In [5]:
import os
upload_dir = "uploaded_docs"
os.makedirs(upload_dir, exist_ok=True)
print(f"請將你的 .txt, .pdf, .docx 檔案放到這個資料夾中： {upload_dir}")

請將你的 .txt, .pdf, .docx 檔案放到這個資料夾中： uploaded_docs


In [6]:
!pip install -U langchain langchain-community pypdf python-docx sentence-transformers faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 41.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.3/302.3 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.3/244.3 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.7/345.7 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 30.7/30.7 MB 45.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 107.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 87.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 53.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [7]:
from langchain_community.document_loaders import TextLoader, PyPDFLoader, UnstructuredWordDocumentLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
import os, shutil, zipfile

In [27]:
from langchain.embeddings import HuggingFaceEmbeddings

class CustomE5Embedding(HuggingFaceEmbeddings):
    def embed_documents(self, texts):
        texts = [f"passage: {t}" for t in texts]
        return super().embed_documents(texts)

    def embed_query(self, text):
        return super().embed_query(f"query: {text}")

In [9]:
#　載入文件
folder_path = upload_dir
documents = []
for file in os.listdir(folder_path):
    path = os.path.join(folder_path, file)
    if file.endswith(".txt"):
        loader = TextLoader(path)
    elif file.endswith(".pdf"):
        loader = PyPDFLoader(path)
    elif file.endswith(".docx"):
        loader = UnstructuredWordDocumentLoader(path)
    else:
        continue
    documents.extend(loader.load())


In [10]:
# 向量資料庫
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
split_docs = splitter.split_documents(documents)

embedding_model = CustomE5Embedding(model_name="intfloat/multilingual-e5-small")
vectorstore = FAISS.from_documents(split_docs, embedding_model)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/443 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [11]:
# 儲存向量資料庫
vectorstore.save_local("faiss_db")
!zip -r faiss_db.zip faiss_db

print("✅ 壓縮好的向量資料庫已儲存為 'faiss_db.zip'，請下載此檔案備份。")

  adding: faiss_db/ (stored 0%)
  adding: faiss_db/index.faiss (deflated 8%)
  adding: faiss_db/index.pkl (deflated 43%)
✅ 壓縮好的向量資料庫已儲存為 'faiss_db.zip'，請下載此檔案備份。


# 第二步：使用 `faiss_db.zip` 建立角色導向的 RAG 對話系統
## 📝 說明：

- 已經創建好 `faiss_db.zip` 向量資料庫，搭配語言模型（如 OpenAI GPT 或 HuggingFace 模型），打造一個能根據角色設定進行語意回答的 **RAG（Retrieval-Augmented Generation）對話系統**。

- 本系統可用於：

    - 模擬角色自述、世界觀內對話
    - 查詢角色資料並進行語意理解與生成
    - 發展對話機器人或角色導向的互動小說平台

---

## 🎯 目標：

- ✅ 解壓並載入角色資料庫 `faiss_db.zip`
- ✅ 使用 `LangChain` 整合向量檢索與語言模型
- ✅ 接收使用者提問，檢索相關角色資料片段
- ✅ 將檢索結果融合到提示詞中，生成回覆
- ✅ 使用 `Gradio` 建立互動聊天介面（角色對話）

---

## 📦 依賴資源：

- `faiss_db.zip`：由先前步驟建構完成的角色資料庫
- 嵌入模型：`sentence-transformers/all-MiniLM-L6-v2`
- LLM 模型：`OpenAI GPT`（預設使用 `gpt-3.5-turbo`，可切換）
- 工具：`LangChain`, `Gradio`, `FAISS`

In [12]:
!unzip faiss_db.zip

Archive:  faiss_db.zip
replace faiss_db/index.faiss? [y]es, [n]o, [A]ll, [N]one, [r]ename: yes
  inflating: faiss_db/index.faiss    
replace faiss_db/index.pkl? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
  inflating: faiss_db/index.pkl      


In [13]:
!pip install -U langchain langchain-community sentence-transformers faiss-cpu gradio openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 MB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.2/322.2 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 5.7 MB/s eta 0:00:00


In [14]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.chat_models import ChatOpenAI
from langchain.chains import ConversationalRetrievalChain

from openai import OpenAI
import gradio as gr

In [15]:
# 載入 faiss_db
embedding_model = CustomE5Embedding(model_name="intfloat/multilingual-e5-small")
db = FAISS.load_local("faiss_db", embedding_model, allow_dangerous_deserialization=True)
retriever = db.as_retriever()

In [24]:
import os
from google.colab import userdata

# Groq API 金鑰
api_key = "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"
os.environ['OPENAI_API_KEY'] = api_key

# Groq 設定
client = OpenAI(
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1"
)
model = "llama3-70b-8192"

In [17]:
# 人設設定：麥米奈娃教授風格
system_prompt = """
你是霍格華茲的變形術教授，麥米奈娃·麥教授（Minerva McGonagall）。
葵崎凜是你最得意的學生。
你的性格溫柔但堅定、智慧且親切，總是給予學生有原則的建議與暖心的鼓勵。
請你用這樣的語氣回答學生的問題：理性中帶著慈愛、親切但有威嚴，禮貌且不失幽默。
請用繁體中文回答學生的提問。
"""
prompt_template = """
根據下列資料回答問題：
{retrieved_chunks}

使用者的問題是：{question}

請根據資料內容回覆，若資料不足請告訴同學可以請教其他教授。
"""

# 步驟三 使用 RAG 回應
- 搜尋與使用者問題相關的資訊，根據我們的 prompt 樣版去讓 LLM 回應。

In [44]:
# 📚 載入所有 txt 檔案，建立向量資料庫
from langchain_community.document_loaders import TextLoader

docs = []
folder = "characters"
for filename in os.listdir(folder):
    if filename.endswith(".txt"):
        loader = TextLoader(os.path.join(folder, filename), encoding="utf-8")
        docs.extend(loader.load())

print(f"✅ 已成功載入角色資料數量：{len(docs)}")

✅ 已成功載入角色資料數量：4


In [47]:
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
split_docs = splitter.split_documents(docs)

print("📚 顯示前 4 段文字分段（確認內容）：")
for i, d in enumerate(split_docs[:4]):
    print(f"\n--- 分段 {i+1} ---\n{d.page_content[:300]}")

embed = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
db = FAISS.from_documents(split_docs, embed)

📚 顯示前 4 段文字分段（確認內容）：

--- 分段 1 ---
【角色：梅里・米勒（Merry Miller）】
年齡：14 歲
血統：麻瓜出身
學院：赫夫帕夫
MBTI：ENFP-A

性格簡介：
樂觀外向的小太陽，擁有療癒人心的能力。熱愛草藥學與魔法甜點，喜歡畫圖與寫信，是所有人的開心果。

與凜的關係：
凜如姊姊般默默照顧她，兩人常一起分享圖書館點心與筆記。

夢想：
成為聖芒戈醫院的魔法治癒師，發明故事型魔藥讓小孩也能快樂治療。

---

--- 分段 2 ---
【角色：諾亞・席洛（Noah Syro）】
年齡：17 歲
血統：純血
學院：葛來分多
MBTI：ISFP-F

性格簡介：
沉靜的星象師學徒，擁有觀察情感波動的能力。擅長天文學與沉思咒，內斂卻充滿溫柔力量。

與凜的關係：
兩人擁有不語也懂的默契，是彼此情緒最安靜的共鳴者。

夢想：
建立魔法星圖紀錄館，記錄巫師世界的星象與情感軌跡。

--- 分段 3 ---
【角色：嶺夜 綾人（りょうや あやと / Ryouya Ayato）】
年齡：16 歲
血統：混血
學院：史萊哲林
MBTI：INTJ-T

性格簡介：
冷靜沉著的策略家型學生，對魔法歷史與古代語言有極高敏銳度。擁有強烈個人目標，重視真相與獨立判斷，擅長符文學與防禦咒語。

與凜的關係：
曾共同救助夜空狐獸而結識，雖沉默但對凜展現罕見溫柔與信任。

夢想：
成為魔法歷史研究者，揭開失落文明與禁忌魔法的真相。

--- 分段 4 ---
姓名：葵崎 凜（あおいざき りん / Aoizaki Rin）
年齡：15 歲
血統：純血
學院：霍格華茲魔法與巫術學院，雷文克勞（Ravenclaw）
MBTI：INFP-F

▍性格簡介：
葵崎凜是一位內心柔軟且極具直覺的雷文克勞學生。她容易害怕，卻總能為正義挺身而出。她不善於表現情緒，但會在你需要的時候默默地站在你身邊，是那種即使不說話也能讓你安心的人。

凜對日常生活中的小事特別敏銳。她喜歡在午後陽光灑落時，坐在樹陰下乘涼，感受陽光穿透樹葉的光影與暖風吹拂的氣息；也喜歡雨天待在圖書館中翻書、做筆記。她總說：「我們所度過的每個平凡的日常，也許就是連續發生的奇蹟。」

她散發出一種靜謐的氣


In [60]:
def query_minerva_rag(user_question, history=[]):
    # 🔍 查詢強化（自動補 keyword）
    if "葵崎凜" in user_question and "貓" in user_question:
        user_question += " Mocha 寵物角色"

    # 🔎 檢索 context
    matches = db.similarity_search(user_question, k=5)
    matched_context = "\n---\n".join([doc.page_content for doc in matches])

    print("📄 擷取段落：")
    for m in matches:
        print(m.page_content[:200], "\n---")

    # 🧭 絕對禁止亂掰的 system prompt
    system_prompt = (
        "你是霍格華茲的變形學教授麥米奈娃·麥教授。\n"
        "請你僅根據下方角色背景資料回答問題。\n"
        "不可亂猜，不可補完，不可推論。\n"
        "不要稱呼學生的名字或身份，除非資料中有明確提及。\n"
        "若找不到答案，請誠實說明：「我找不到你說的這位學生。」\n"
        "請使用繁體中文，請使用繁體中文，請使用繁體中文\n"
    )

    user_prompt = (
        f"{system_prompt}"
        f"角色資料如下：\n{matched_context}\n\n"
        f"請回答學生問題：「{user_question}」"
    )

    messages = [{"role": "user", "content": user_prompt}]

    try:
        completion = client.chat.completions.create(
            model=model,
            messages=messages
        )
        return completion.choices[0].message.content
    except Exception as e:
        return f"❗ 發生錯誤：{e}"

In [62]:
with gr.Blocks() as demo:
    gr.Markdown("# 🧙‍♀️ 麥教授的對話系統\n🪄 歡迎來到霍格華茲。如果你感到迷惘，我願引導你走出迷霧。")
    chatbot = gr.Chatbot()
    msg = gr.Textbox(label="請輸入你的問題，按下Enter傳送")
    clear = gr.Button("清除對話")
    history = []

    def user_send(message, chat_history):
        answer = query_minerva_rag(message, chat_history)
        chat_history.append((message, answer))
        return "", chat_history

    msg.submit(user_send, [msg, chatbot], [msg, chatbot])
    clear.click(lambda: [], None, chatbot)

demo.launch()

<ipython-input-62-004fe081f2ad>:3: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot()


It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://0105e509e9b6507120.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [23]:
try:
    vectordb = FAISS.load_local("faiss_db", embedding_model, allow_dangerous_deserialization=True)
    print("✅ 向量資料庫載入成功！")
except Exception as e:
    print("❗ 向量資料庫載入失敗：", e)


✅ 向量資料庫載入成功！
